# Q1 SaafHawa calculations
Riya Bhart | 23k0063
Run from the Q1 folder or the codes folder. Costs with unknown cache eligibility are conservatively identical to uncached; zero added reasoning is a lower bound where reasoning is mandatory. Endpoints and evidence gaps are listed in the report. Run the chart cell with matplotlib.

In [1]:
rows=[('gpt-5.4-nano', 'OpenAI / Standard', 'Not disclosed', '400k / 128k', 0.2, 1.25, 0.02, 1408, 0.2, 'Unverified', '0.87 / 156.4', 'Tool score unverified', 'Tools + JSON; none; proprietary', 'A M O'), ('gpt-5.4-mini', 'OpenAI / Standard', 'Not disclosed', '400k / 128k', 0.75, 4.5, 0.075, 1408, 0.75, 'Unverified', 'Unverified', 'Unverified', 'Tools + JSON; none; proprietary', 'B O'), ('claude-haiku-5-5', 'Anthropic / Claude API paid', 'Not disclosed', '1M / 128k', 0.1, 0.5, 0.01, 1500, 0.125, 'Tier 1 unverified', 'Unverified', 'Unverified', 'Tools + JSON; disabled where supported; proprietary', 'C P X'), ('claude-sonnet-5-5', 'Anthropic / Claude API paid', 'Not disclosed', '1M / 128k', 2, 10, 0.1, 1500, 2.5, 'Tier 1 unverified', 'Unverified', 'Unverified', 'Tools + JSON; disable thinking; proprietary', 'D P X'), ('gemini-3.6-flash', 'Google / Developer API paid', 'Not disclosed', '1,048,576 / 65,536', 0.75, 3.75, 0.075, 0, 0.75, 'Account-specific; unverified', 'Unverified', 'Unverified', 'Tools + JSON; minimal; proprietary', 'E T'), ('grok-4.3', 'xAI / Standard T0', 'Not disclosed', '1M / unverified', 1.25, 2.5, 0.2, 0, 1.25, '2,220 RPM equiv.; 10M TPM', '0.71 / 100.1', 'AA Index 14; tool score unverified', 'Tools + JSON; none; proprietary; legacy', 'F N Q'), ('gpt-oss-20b', 'OpenAI / Groq Developer', '20.9B / 3.6B', '131,072 / 65,536', 0.075, 0.3, None, 0, 0.075, '1,000 RPM; 250k TPM', 'Unverified / ~1,000', 'Tau-Bench Retail 35.0%; MMLU 80.4% (low)', 'Tools + JSON; low; Apache 2.0', 'G H Y'), ('gpt-oss-120b', 'OpenAI / Groq Developer', '116.8B / 5.1B', '131,072 / 65,536', 0.15, 0.6, None, 0, 0.15, '1,000 RPM; 250k TPM', 'Unverified / ~500', 'Tau-Bench Retail 49.4%; MMLU 85.9% (low)', 'Tools + JSON; low; Apache 2.0', 'G H'), ('qwen/qwen3-8b', 'Alibaba / OpenRouter paid', '8.2B dense', '40,960 native / unverified', 0.117, 0.455, None, 0, 0.117, 'Route-dependent', 'Unverified', 'Unverified', 'Non-thinking; endpoint tool/JSON varies; Apache 2.0', 'I R Z'), ('qwen/qwen3-30b-a3b-instruct-2507', 'Alibaba / OpenRouter paid', '30.5B / 3.3B', '262k / unverified', 0.09, 0.3, None, 0, 0.09, 'Route-dependent', 'Unverified', 'Unverified', 'Non-thinking; endpoint tool/JSON varies; Apache 2.0', 'J U'), ('z-ai/glm-4.5-air', 'Z.ai / NovitaAI via OpenRouter', '106B / 12B', '131,072 / 98,304', 0.13, 0.85, 0.025, 0, 0.13, 'Route-dependent', '1.49 / 36 (route P50)', '12-test composite 59.8; per-task unverified', 'Thinking off; tools yes, strict JSON no; MIT', 'K V'), ('deepseek/deepseek-v4-flash', 'DeepSeek / OpenRouter paid', '284B / 13B', '1M / unverified', 0.0474, 0.45, 0.0474, 0, 0.0474, 'Route-dependent', 'Unverified', 'Unverified', 'Non-thinking where supported; tool/JSON varies; MIT', 'L W')]

In [2]:

from math import ceil,floor
from pathlib import Path
import csv
from statistics import median
R=63
D=sum(int(x) for x in '23k0063' if x.isdigit())%10
Q,N,p,H,B,T=1500+25*R,5+R%4,.04+.01*D,4+R%4,1200+10*R,10-R%3
assert (D,Q,N,p,H,B,T)==(4,3075,8,.08,7,1830,10)
inputs=[2000+800*k for k in range(N)]
outputs=[350]*(N-1)+[600]
tries=[1+p]*(N-1)+[1]
I=sum(a*b for a,b in zip(inputs,tries))
O=sum(a*b for a,b in zip(outputs,tries))
A=sum(tries)
factor=30+4*H
monthly=Q*factor
peak=5*Q*.35/120
print('parameters',R,D,Q,N,p,H,B,T)
print('calls (input, output, attempts)',list(zip(inputs,outputs,tries)))
print('per query',I,O,A,'monthly',monthly,I*monthly,O*monthly)
print('peak qpm/rpm/input tpm/output tpm/combined',peak,peak*A,peak*I,peak*O,peak*(I+O))
def cost(row, cached=False, reasoning=0, volume=Q):
    ri,ro,read,prefix,write=row[4:9]
    prefix=prefix if cached else 0
    # One miss/write per query. Shared cross-query hits and additional misses are possible.
    return volume*factor*((I-A*prefix)*ri+prefix*write+prefix*(A-1)*(read or 0)+(O+A*reasoning)*ro)/1e6
for row in rows:
    print(row[0],*[round(cost(row,c),2) for c in (False,True)])
timing=next((x for x in (Path('../data/tool_timings.csv'),Path('data/tool_timings.csv')) if x.exists()))
with timing.open(encoding='utf8') as f: successful=[r for r in csv.DictReader(f) if r['status']=='ok']
counts={k:(len(s:=[r for r in successful if r['tool']==k]),len({r['timestamp_pkt'][:10] for r in s})) for k in ('air','weather','heat_index','kb')}
provisional=any(counts[k][0]<20 or counts[k][1]<2 for k in ('air','weather'))
m=median(float(r['seconds']) for r in successful)
print('timing coverage, provisional, pooled median',counts,provisional,m)
print('API-only median',median(float(r['seconds']) for r in successful if r['tool'] in ('air','weather')))
def latency(ttft,speed):
    # Fit an illustrative affine TTFT curve anchored at 4,800 input tokens.
    # Keep 60% of that benchmark TTFT fixed and scale the remaining 40% with input.
    first=lambda n:ttft*(.6+.4*n/4800)
    step=[first(i)+(o-1)/speed for i,o in zip(inputs,outputs)]
    retry=p*sum(step[:-1])
    # Redesigned loop: one structured extraction then one answer.
    # Heat follows weather. A planning retry occurs with probability p.
    revised=(1+p)*(first(2000)+99/speed)+first(3600)+179/speed+2*m
    return sum(step)+retry+7*m,revised,retry,step
nano=latency(.87,156.4)
grok=latency(.71,100.1)
print('nano and grok: full, redesigned, retries, per-step',nano,grok)
length=max(i+o for i,o in zip(inputs,outputs))
kv=2*36*8*128*length*2
weights=[8.2e9*b for b in (2,1,.5)]
capacity=floor((.9*24*2**30-weights[-1])/kv)
# Use the 10 s service objective for provisioning. A slower live service needs more slots.
def gpus(q,seconds=T):
    return ceil(5*q*.35/120*seconds/60/capacity)
def host(q): return 720*.34*gpus(q)
unit=cost(rows[0],True,volume=1)
k=5*.35/120*T/60/capacity
floor_slope=720*.34*k
# If the asymptotic hosting slope is >= API slope, hosting never wins.
# Otherwise the first feasible integer lies before the first guaranteed crossing.
qstar=None if floor_slope>=unit else next(q for q in range(1,ceil(720*.34/(unit-floor_slope))+2) if host(q)<q*unit)
print('seq length, weights GB, KV GiB, slots',length,[w/1e9 for w in weights],kv/2**30,capacity)
print('concurrent at target, GPUs, host USD',peak*T/60,gpus(Q),host(Q))
print('concurrent at full nano latency, GPUs',peak*nano[0]/60,gpus(Q,nano[0]))
print('API slope, GPU slope floor, break-even',unit,floor_slope,qstar)
new_input=2000*(1+p)+3600
new_output=100*(1+p)+180
new_api=monthly*(new_input*.20+new_output*1.25)/1e6
print('Redesigned nano monthly uncached estimate',new_api)


parameters 63 4 3075 8 0.08 7 1830 10
calls (input, output, attempts) [(2000, 350, 1.08), (2800, 350, 1.08), (3600, 350, 1.08), (4400, 350, 1.08), (5200, 350, 1.08), (6000, 350, 1.08), (6800, 350, 1.08), (7600, 600, 1)]
per query 40864.0 3246.0 8.56 monthly 178350 7288094400.0 578924100.0
peak qpm/rpm/input tpm/output tpm/combined 44.84375 383.8625 1832495.0 145562.8125 1978057.8125
gpt-5.4-nano 2181.27 1839.55
gpt-5.4-mini 8071.23 6789.78
claude-haiku-5-5 1018.27 842.94
claude-sonnet-5-5 20365.43 16656.46
gemini-3.6-flash 7637.04 7637.04
grok-4.3 10557.43 10557.43
gpt-oss-20b 720.28 720.28
gpt-oss-120b 1440.57 1440.57
qwen/qwen3-8b 1116.12 1116.12
qwen/qwen3-30b-a3b-instruct-2507 829.61 829.61
z-ai/glm-4.5-air 1439.54 1439.54
deepseek/deepseek-v4-flash 605.97 605.97
timing coverage, provisional, pooled median {'air': (20, 2), 'weather': (20, 2), 'heat_index': (20, 2), 'kb': (20, 2)} False 0.2521454499801621
API-only median 1.6144481500377879
nano and grok: full, redesigned, retries, p

In [ ]:
import matplotlib.pyplot as plt
x=sorted({*range(0,60001,25),*[int(j/k)+a for j in range(1,20) for a in (0,1) if 0<int(j/k)+a<=60000]})
plt.plot(x,[unit*q for q in x],label='Nano API with modeled caching')
plt.step(x,[host(q) for q in x],where='post',label='RTX 4090 self-hosting')
plt.axvline(Q,color='gray',linestyle=':',label='Assignment volume')
plt.xlabel('Normal-day queries')
plt.ylabel('USD per 30 days')
plt.legend()
plt.tight_layout()
plt.show()
